**Виконавець:** Базилевич Олексій, група К-27
**Викладач:** Андрій Ляшко

**Умова задачі:**
Написати одновіконну програму під Plotly&Dash.
Залежно від періоду та графіку показати:
а) графік зміни денної та нічної температури
б) графік хмарності
в) графік сили вітру
г) бульбашковий графік денної температури від опадів.
Залежно від обраної аналітики:
а) гістограма відхилення нічної температури від денної
б) стовпчикова діаграма хмарності з накопиченням
в) діаграма «сонячного вибуху» хмарності
г) кругова діаграма днів з опадами.

In [1]:
import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go

In [2]:
def process_data(file_path):
    df = pd.read_csv(file_path, encoding="UTF-8")

    df['хмарність'] = df['хмарність'].str.replace('%', '').astype(int)

    df['опади'] = df['опади'].replace('-', None).fillna('0 м.м.')
    df['опади_num'] = df['опади'].str.replace(' м.м.', '').astype(float)

    df['денна температура повітря'] = df['денна температура повітря'].str.replace('°C', '').astype(float)
    df['нічна температура повітря'] = df['нічна температура повітря'].str.replace('°C', '').astype(float)
    df['сила вітру'] = df['сила вітру'].str.replace(' м/с', '').astype(float)

    df['відхилення'] = df['нічна температура повітря'] - df['денна температура повітря']

    df['Хмарність_категорія'] = pd.cut(df['хмарність'], bins=[-1, 34, 70, 100], labels=['сонячний', 'мінлива хмарність', 'хмарний'])

    df['Є_опади'] = df['опади_num'] > 0
    df['Розмір_бульбашки'] = df['опади_num'] * 5
    df.loc[df['опади_num'] == 0, 'Розмір_бульбашки'] = 2

    return df

In [3]:
df = process_data('weather2026.csv')

In [4]:
app = Dash(__name__)
app.config.suppress_callback_exceptions = True

graph_menu = dcc.Dropdown(
    id='graph-dropdown',
    options=[
        {'label': 'Температура (денна/нічна)', 'value': 'temp'},
        {'label': 'Хмарність', 'value': 'clouds'},
        {'label': 'Сила вітру', 'value': 'wind'},
        {'label': 'Денна температура', 'value': 'scatter'}
    ],
    value='temp',
    clearable=False,
    style={'width': '100%', 'fontSize': '18px'}
)

period_menu = dcc.Dropdown(
    id='period-dropdown',
    options=[{'label': str(p), 'value': str(p)} for p in df['період'].unique()],
    value=str(df['період'].unique()[0]),
    clearable=False,
    style={'width': '100%', 'fontSize': '18px'}
)

analytics_menu = dcc.Dropdown(
    id='analytics-dropdown',
    options=[
        {'label': 'Відхилення температур', 'value': 'hist'},
        {'label': 'Типи днів', 'value': 'bar'},
        {'label': 'Розподіл хмарності', 'value': 'sunburst'},
        {'label': 'Дні з опадами', 'value': 'pie'}
    ],
    value='hist',
    clearable=False,
    style={'width': '100%', 'fontSize': '18px'}
)

app.layout = html.Div([
    html.H1('Аналіз погодних даних', style={'textAlign': 'center', 'color': '#503D36'}),

    html.Div([
        html.Div([
            html.H2('Графік:', style={'marginRight': '10px'}),
            html.Div(graph_menu, style={'flex': '1'})
        ], style={'display': 'flex', 'alignItems': 'center', 'width': '45%'}),

        html.Div([
            html.H2('Місяць:', style={'marginRight': '10px'}),
            html.Div(period_menu, style={'flex': '1'})
        ], style={'display': 'flex', 'alignItems': 'center', 'width': '45%'}),
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '20px'}),

    dcc.Graph(id='main-graph'),

    html.H1('Аналітика за весь період', style={'textAlign': 'center', 'color': '#503D36', 'marginTop': '30px'}),

    html.Div([
        html.H2('Тип аналітики:', style={'marginRight': '10px'}),
        html.Div(analytics_menu, style={'flex': '1'})
    ], style={'display': 'flex', 'alignItems': 'center', 'width': '80%', 'margin': '0 auto', 'marginBottom': '20px'}),

    dcc.Graph(id='analytics-graph')

], style={'padding': '20px'})

@app.callback(
    Output('main-graph', 'figure'),
    [Input('period-dropdown', 'value'), Input('graph-dropdown', 'value')]
)
def update_main_graph(period, graph_type):
    if not period or not graph_type:
        return go.Figure()

    dff = df[df['період'] == period]

    if dff.empty:
        return go.Figure()

    if graph_type == 'temp':
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=dff['день'], y=dff['денна температура повітря'], name='Денна', mode='lines+markers'))
        fig.add_trace(go.Scatter(x=dff['день'], y=dff['нічна температура повітря'], name='Нічна', mode='lines+markers'))
        return fig
    elif graph_type == 'clouds':
        return px.line(dff, x='день', y='хмарність', markers=True)
    elif graph_type == 'wind':
        return px.line(dff, x='день', y='сила вітру', markers=True)
    elif graph_type == 'scatter':
        return px.scatter(dff, x='день', y='денна температура повітря', size='Розмір_бульбашки')

@app.callback(
    Output('analytics-graph', 'figure'),
    [Input('analytics-dropdown', 'value')]
)
def update_analytics_graph(analytics_type):
    if not analytics_type:
        return go.Figure()

    if analytics_type == 'hist':
        return px.histogram(df, x='відхилення')
    elif analytics_type == 'bar':
        grouped = df.groupby(['період', 'Хмарність_категорія'], observed=False).size().reset_index(name='count')
        return px.bar(grouped, x='період', y='count', color='Хмарність_категорія')
    elif analytics_type == 'sunburst':
        grouped = df.groupby(['період', 'Хмарність_категорія'], observed=False).size().reset_index(name='count')
        return px.sunburst(grouped, path=['період', 'Хмарність_категорія'], values='count')
    elif analytics_type == 'pie':
        grouped = df.groupby('період', observed=False)['Є_опади'].sum().reset_index(name='days_with_rain')
        return px.pie(grouped, values='days_with_rain', names='період')

if __name__ == '__main__':
    app.run(jupyter_mode='inline', jupyter_height=1000)